# MedSAM-Agent3D — Google Colab Runner

Trains an MLLM (Qwen3-VL-8B-Instruct) to use SAM-Med3D for interactive 3D medical image segmentation.

**Runtime type:** T4 GPU (or better)

## Workflow
1. Mount Google Drive & install dependencies
2. Clone the MedSAM-Agent3D repo
3. Download model checkpoints
4. Run the data pipeline (dummy or TotalSegmentator)
5. (Optional) SFT + RL training

---
## 1. Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create a working directory in Drive
import os
WORK_DIR = '/content/drive/MyDrive/MedSAM-Agent3D'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {WORK_DIR}')

In [ ]:
# Install system dependencies
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 > /dev/null 2>&1

# Install Python packages
!pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 \
    transformers>=4.45.0 \
    accelerate \
    trl>=0.9.0 \
    nibabel \
    scipy \
    opencv-python \
    Pillow \
    scikit-image \
    tqdm \
    datasets \
    huggingface-hub \
    bitsandbytes \
    peft \
    qwen-vl-utils

print('\n✅ All dependencies installed')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---
## 2. Clone MedSAM-Agent3D Repo

In [ ]:
import os, shutil

REPO_DIR = os.path.join(WORK_DIR, 'MedSAM-Agent3D')
if os.path.exists(REPO_DIR):
    print('Repo already exists. Pulling latest...')
    %cd $REPO_DIR
    !git pull
else:
    %cd $WORK_DIR
    !git clone https://github.com/your-org/MedSAM-Agent3D.git 2>/dev/null || \
     echo 'Git clone failed — creating directory structure manually'
    if not os.path.exists(REPO_DIR):
        # Fallback: copy from Drive upload
        print('Please upload the MedSAM-Agent3D folder to your Drive.')
        print('Then run this cell again.')

%cd $REPO_DIR
print(f'\nRepo at: {REPO_DIR}')

---
## 3. Download Model Checkpoints

This step downloads:
- **Qwen3-VL-8B-Instruct** (~16 GB) — the MLLM backbone for SFT/RL
- **SAM-Med3D Turbo** checkpoint — the 3D segmentation backbone

In [ ]:
MODEL_CACHE = os.path.join(WORK_DIR, 'model_cache')
os.makedirs(MODEL_CACHE, exist_ok=True)

from huggingface_hub import snapshot_download

# Download Qwen3-VL-8B-Instruct
QWEN_DIR = os.path.join(MODEL_CACHE, 'Qwen3-VL-8B-Instruct')
if not os.path.exists(os.path.join(QWEN_DIR, 'config.json')):
    print('Downloading Qwen3-VL-8B-Instruct (~16 GB)...')
    snapshot_download(
        repo_id='Qwen/Qwen3-VL-8B-Instruct',
        local_dir=QWEN_DIR,
        local_dir_use_symlinks=False,
        resume_download=True
    )
    print('✅ Qwen3-VL-8B-Instruct downloaded')
else:
    print('✅ Qwen3-VL-8B-Instruct already cached')

In [ ]:
# Download SAM-Med3D Turbo checkpoint
SAM_CKPT = os.path.join(REPO_DIR, 'sam_med3d_turbo.pth')
if not os.path.exists(SAM_CKPT):
    print('Downloading SAM-Med3D Turbo checkpoint...')
    !wget -qO $SAM_CKPT https://huggingface.co/your-org/sam-med3d-turbo/resolve/main/sam_med3d_turbo.pth 2>/dev/null || \
      echo 'WARNING: SAM-Med3D checkpoint URL not available. Using mock wrapper.'
    if os.path.exists(SAM_CKPT) and os.path.getsize(SAM_CKPT) > 1e6:
        print(f'✅ SAM-Med3D checkpoint saved ({os.path.getsize(SAM_CKPT)/1e6:.0f} MB)')
else:
    print(f'✅ SAM-Med3D checkpoint already exists ({os.path.getsize(SAM_CKPT)/1e6:.0f} MB)')

---
## 4. Run Data Pipeline

Generates synthetic (dummy) data or processes real TotalSegmentator MRI data.

In [ ]:
# ─── Configuration ───
MODE = 'dummy'  # 'dummy' or 'totalsegmentator'
MAX_SAMPLES = 3  # Number of volumes to process
SEED = 42
# ─────────────────────

%cd $REPO_DIR

if MODE == 'dummy':
    !python run_data_pipeline.py --mode dummy --seed $SEED
elif MODE == 'totalsegmentator':
    # Ensure TotalSegmentator data is in the right place
    TS_DIR = os.path.join(REPO_DIR, 'data', 'totalsegmentator')
    os.makedirs(TS_DIR, exist_ok=True)
    !python run_data_pipeline.py --mode totalsegmentator --ts_dir $TS_DIR --max_samples $MAX_SAMPLES --seed $SEED

print('\n✅ Data pipeline complete!')

In [ ]:
# Inspect generated data
import json, os

print('=== Trajectories ===')
traj_dir = os.path.join(REPO_DIR, 'data', 'trajectories')
if os.path.exists(traj_dir):
    for f in os.listdir(traj_dir):
        if f.endswith('.json'):
            with open(os.path.join(traj_dir, f)) as fh:
                traj = json.load(fh)
            print(f'  {f}: {len(traj)} turns')

print(f'\n=== Images ===')
img_dir = os.path.join(REPO_DIR, 'data', 'images')
if os.path.exists(img_dir):
    pngs = [f for f in os.listdir(img_dir) if f.endswith('.png')]
    print(f'  {len(pngs)} PNG files')

print(f'\n=== SFT Dataset ===')
sft_path = os.path.join(REPO_DIR, 'data', 'sft_dataset.jsonl')
if os.path.exists(sft_path):
    with open(sft_path) as f:
        count = sum(1 for _ in f)
    print(f'  {count} SFT samples')

---
## 5. SFT Training (Optional)

Fine-tunes Qwen3-VL-8B-Instruct on the generated trajectories.

**Requirements:** ~24 GB VRAM (T4 with gradient checkpointing + LoRA)

In [ ]:
# ─── Configuration ───
RUN_SFT = False  # Set to True to run SFT training
SFT_OUTPUT = os.path.join(REPO_DIR, 'models', 'sft_output')
# ─────────────────────

if RUN_SFT:
    print('Starting SFT training...')
    
    # Use 4-bit QLoRA to fit in T4 memory
    !python -c """
import sys
sys.path.insert(0, '{REPO_DIR}')

from training.sft_trainer import train_sft
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

train_sft(
    dataset_path='{REPO_DIR}/data/sft_dataset.jsonl',
    model_id='{QWEN_DIR}',
    output_dir='{SFT_OUTPUT}',
    seed=42
)
"""
    print(f'✅ SFT training complete. Model saved to {SFT_OUTPUT}')
else:
    print('Skipping SFT training. Set RUN_SFT = True to enable.')

---
## 6. RL Fine-Tuning (Optional)

Reinforcement learning phase using PPO on top of the SFT model.

**Requirements:** ~32 GB VRAM (A100 recommended for full training)

In [ ]:
# ─── Configuration ───
RUN_RL = False  # Set to True to run RL training
RL_OUTPUT = os.path.join(REPO_DIR, 'models', 'rl_output')
# ─────────────────────

if RUN_RL:
    print('Starting RL fine-tuning...')
    
    !python -c """
import sys
sys.path.insert(0, '{REPO_DIR}')

from training.rl_trainer import train_rl

train_rl(
    sft_model_path='{SFT_OUTPUT if RUN_SFT else QWEN_DIR}',
    output_dir='{RL_OUTPUT}'
)
"""
    print(f'✅ RL training complete. Model saved to {RL_OUTPUT}')
else:
    print('Skipping RL training. Set RUN_RL = True to enable.')

---
## 7. Visualize Sample Trajectory

See the tri-view images the model will train on.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os, json

# Load a sample trajectory
traj_dir = os.path.join(REPO_DIR, 'data', 'trajectories')
traj_files = [f for f in os.listdir(traj_dir) if f.endswith('.json')]

if traj_files:
    with open(os.path.join(traj_dir, traj_files[0])) as f:
        traj = json.load(f)
    
    n_turns = len(traj)
    fig, axes = plt.subplots(1, n_turns, figsize=(5*n_turns, 5))
    if n_turns == 1:
        axes = [axes]
    
    for i, (turn_key, turn_data) in enumerate(traj.items()):
        img_path = os.path.join(REPO_DIR, turn_data['image'])
        img = Image.open(img_path)
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"{turn_key}\n{turn_data['action'].get('name', '')}")
        axes[i].axis('off')
    
    plt.suptitle(f"Trajectory: {traj_files[0]}", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print the actions
    print('Actions:')
    for turn_key, turn_data in traj.items():
        action = turn_data['action']
        if 'point_3d' in action:
            print(f"  {turn_key}: {action['name']} at {action['point_3d']} ({action['type']})")
        else:
            print(f"  {turn_key}: {action['name']}")
else:
    print('No trajectories found. Run the data pipeline first.')

---
## Summary

| Step | Command | Status |
|---|---|---|
| Data pipeline | `run_data_pipeline.py --mode dummy` | ✅ |
| SFT training | `sft_trainer.py --model Qwen/Qwen3-VL-8B-Instruct` | 🟡 Set `RUN_SFT = True` |
| RL training | `rl_trainer.py --model models/sft_output` | 🟡 Set `RUN_RL = True` |

**All outputs are saved to your Google Drive** at `MyDrive/MedSAM-Agent3D/`.

### Next Steps
1. Upload `sam_med3d_turbo.pth` to the repo directory for real SAM-Med3D integration
2. Upload TotalSegmentator MRI data to `data/totalsegmentator/` if using real data
3. Enable SFT training with proper LoRA config for T4 GPU
4. Run inference on new volumes using the fine-tuned model